# FedRep Cleint

> The core abstraction for `FedRep` client: [https://proceedings.mlr.press/v139/collins21a.html](https://proceedings.mlr.press/v139/collins21a.html)

In [ ]:
#| default_exp clients.fedrep

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import copy
import random
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from fedai.clients.base_client import BaseClient
from fedai.utils.registery import AlgorithmRegistry

<torch._C.Generator>

In [ ]:
#| export
@AlgorithmRegistry.register_client("fedrep")
class ClientFedRep(BaseClient):
    def __init__(self,
                 id, # Unique identifier for the client
                 cfg, # A configuration object containing all the necessary parameters for training and evaluation.
                 train_loader, 
                 test_loader,
                 state, # A dictionary containing the model, optimizer and any changing component needed.
                 criterion,
                 device= None,
                 t= 0,
                 **kwargs # extra client-specific parameters (that cannot be passed in state and cfg), can be passed as here.
                 ):  
                 
        super().__init__(id, cfg, train_loader, test_loader, state, criterion, device, t, **kwargs)
        

### Training

In [ ]:
#| export
@patch
def fit(self: ClientFedRep):
    
    self.backbone.to(self.device)
    self.head.to(self.device)
    self.backbone.train()
    self.head.train()
    
    for param in self.backbone.parameters():
        param.requires_grad = False
    for param in self.head.parameters():
        param.requires_grad = True

    for _ in range(self.cfg.algorithm.plocal_epochs):
        for i, batch in enumerate(self.train_loader):
            self.optimizer.zero_grad()

            batch = self.send_to_device(batch)
            X, y = batch[self.data_key], batch[self.label_key]
            h = self.backbone(X)
            logits = self.head(h)

            loss = self.criterion(logits, y)
            loss.backward()
            self.optimizer.step()

    for param in self.backbone.parameters():
        param.requires_grad = True
    for param in self.head.parameters():
        param.requires_grad = False


    for _ in range(self.cfg.local_epochs):
        for i, batch in enumerate(self.train_loader):
            self.optimizer.zero_grad()

            batch = self.send_to_device(batch)
            X, y = batch[self.data_key], batch[self.label_key]
            h = self.backbone(X)
            logits = self.head(h)

            loss = self.criterion(logits, y)
            loss.backward()
            self.optimizer.step()


### Evaluation

In [ ]:
#| export
@patch
def train_test_stats(self: ClientFedRep, batch: dict) -> tuple:
    metrics = {k: 0 for k in list(self.cfg.training_metrics)}  # Ensure metrics is always defined

    X, y = batch[self.data_key], batch[self.label_key]
    h = self.backbone(X)
    logits = self.head(h)
    loss = self.criterion(logits, y)
    y_pred = logits.argmax(dim=-1)
    y_true = batch[self.label_key]

    metrics = self.training_metrics.compute(y_pred= y_pred, y_true= y_true)

    return loss, metrics


In [ ]:
#| export
@patch
def evaluate_local(self: ClientFedRep, loader= 'train') -> dict:
    total_loss = 0
    lst_metrics = []

    self.backbone = self.backbone.to(self.device)
    self.backbone.eval()
    self.head = self.head.to(self.device)
    self.head.eval()
    num_eval = 0
    data_loader = self.train_loader if loader == 'train' else self.test_loader
    
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            batch = self.send_to_device(batch)
            loss, metrics = self.train_test_stats(batch)                 
            if not math.isnan(loss.item()):
                total_loss += loss.item()  
                num_eval += len(batch[self.data_key])  # Ensure num_eval is updated
                lst_metrics.append(metrics)           
    
    avg_loss = total_loss / num_eval if num_eval > 0 else 0.0
    self.logger.info(f"Average {loader} Loss is : {avg_loss}")
    
    if lst_metrics:
        total_metrics = {k: sum(m.get(k, 0) for m in lst_metrics) / len(lst_metrics) for k in self.cfg.test_metrics}
    else:
        total_metrics = {k: 0.0 for k in self.cfg.test_metrics}

    self.logger.info(f"Average {loader} Metrics: {total_metrics}")
    return {"loss": avg_loss, "metrics": total_metrics}


### Saving State

In [ ]:
#| export
@patch
def save_state(self: ClientFedRep, save_to_disk= False):  

    state_dict = {}
    state_dict['backbone'] = self.backbone.state_dict()
    state_dict['head'] = self.head.state_dict()
    state_dict['optimizer'] = self.optimizer.state_dict()

    if save_to_disk:
        state_path = os.path.join(self.cfg.server.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
        if not os.path.exists(os.path.dirname(state_path)):
            os.makedirs(os.path.dirname(state_path))

        torch.save(state_dict, state_path)

    return state_dict

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()